In [ ]:
# Frequency Domain Analysis in Audio Signal Processing
# This notebook explores key concepts in frequency domain analysis including:
# 1. Fourier Series Visualization
# 2. Fourier Transform Spectrum Analysis
# 3. Discrete Fourier Transform (DFT) Implementation
# 4. Fast Fourier Transform (FFT) Visualization and Comparison

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from audiodsp import write_wav
from scipy import signal
from scipy.fft import fft, fftfreq
from IPython import display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
import time

# Set up matplotlib for inline plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['figure.dpi'] = 100

print("Libraries imported successfully!")

In [ ]:
# ===================================================================
# TOPIC 1: FOURIER SERIES VISUALIZATION
# ===================================================================

class FourierSeriesVisualizer:
    def __init__(self, sample_rate=44100, duration=2.0):
        self.sample_rate = sample_rate
        self.duration = duration
        self.t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    def generate_periodic_signal(self, signal_type='square', frequency=1.0, amplitude=1.0):
        """Generate periodic signals for Fourier series analysis"""
        omega = 2 * np.pi * frequency
        
        if signal_type == 'square':
            # Square wave using sign of sine
            y = amplitude * np.sign(np.sin(omega * self.t))
        elif signal_type == 'sawtooth':
            # Sawtooth wave
            y = amplitude * signal.sawtooth(omega * self.t)
        elif signal_type == 'triangle':
            # Triangle wave
            y = amplitude * signal.sawtooth(omega * self.t, 0.5)
        elif signal_type == 'half_rectified':
            # Half-wave rectified sine
            sine_wave = np.sin(omega * self.t)
            y = amplitude * np.where(sine_wave > 0, sine_wave, 0)
        elif signal_type == 'full_rectified':
            # Full-wave rectified sine
            y = amplitude * np.abs(np.sin(omega * self.t))
        else:
            # Default to sine wave
            y = amplitude * np.sin(omega * self.t)
        
        return y
    
    def fourier_series_coefficients(self, signal_type, frequency, n_harmonics):
        """Calculate Fourier series coefficients for common periodic signals"""
        coefficients = {}
        
        if signal_type == 'square':
            # Square wave: only odd harmonics
            coefficients['a0'] = 0
            coefficients['an'] = [0] * n_harmonics
            coefficients['bn'] = [4/(np.pi*n) if n % 2 == 1 else 0 for n in range(1, n_harmonics+1)]
        
        elif signal_type == 'sawtooth':
            # Sawtooth rising from -1 to 1 (scipy.signal.sawtooth): all harmonics, sine terms only
            coefficients['a0'] = 0
            coefficients['an'] = [0] * n_harmonics
            coefficients['bn'] = [-2/(np.pi*n) for n in range(1, n_harmonics+1)]
        
        elif signal_type == 'triangle':
            # Triangle starting at -1 (scipy.signal.sawtooth, width 0.5): odd harmonics, cosine terms only
            coefficients['a0'] = 0
            coefficients['an'] = [-8/(np.pi**2 * n**2) if n % 2 == 1 else 0 for n in range(1, n_harmonics+1)]
            coefficients['bn'] = [0] * n_harmonics
        
        elif signal_type == 'half_rectified':
            # Half-wave rectified sine
            coefficients['a0'] = 1/np.pi
            coefficients['an'] = [2/(np.pi*(1-n**2)) if n % 2 == 0 and n != 1 else 0 for n in range(1, n_harmonics+1)]
            coefficients['bn'] = [0.5 if n == 1 else 0 for n in range(1, n_harmonics+1)]
        
        elif signal_type == 'full_rectified':
            # Full-wave rectified sine
            coefficients['a0'] = 2/np.pi
            coefficients['an'] = [4/(np.pi*(1-n**2)) if n % 2 == 0 else 0 for n in range(1, n_harmonics+1)]
            coefficients['bn'] = [0] * n_harmonics
        
        return coefficients
    
    def reconstruct_from_fourier_series(self, signal_type, frequency, n_harmonics):
        """Reconstruct signal from Fourier series coefficients"""
        coeffs = self.fourier_series_coefficients(signal_type, frequency, n_harmonics)
        omega = 2 * np.pi * frequency
        
        # Start with DC component
        reconstructed = coeffs['a0'] * np.ones_like(self.t)
        
        # Add harmonic components
        for n in range(1, n_harmonics + 1):
            an = coeffs['an'][n-1]
            bn = coeffs['bn'][n-1]
            reconstructed += an * np.cos(n * omega * self.t) + bn * np.sin(n * omega * self.t)
        
        return reconstructed
    
    def plot_fourier_series_analysis(self, signal_type='square', frequency=1.0, n_harmonics=10):
        """Interactive visualization of Fourier series analysis"""
        
        # Generate original signal
        original = self.generate_periodic_signal(signal_type, frequency)
        
        # Reconstruct using Fourier series
        reconstructed = self.reconstruct_from_fourier_series(signal_type, frequency, n_harmonics)
        
        # Get coefficients for spectrum plot
        coeffs = self.fourier_series_coefficients(signal_type, frequency, n_harmonics)
        
        # Create subplots
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # Time domain comparison (show first 3 periods)
        period = 1/frequency
        t_plot_samples = int(3 * period * self.sample_rate)
        t_plot = self.t[:t_plot_samples]
        
        axes[0, 0].plot(t_plot, original[:t_plot_samples], 'b-', linewidth=2, label='Original Signal')
        axes[0, 0].plot(t_plot, reconstructed[:t_plot_samples], 'r--', linewidth=2, label=f'Fourier Series (N={n_harmonics})')
        axes[0, 0].set_xlabel('Time (s)')
        axes[0, 0].set_ylabel('Amplitude')
        axes[0, 0].set_title(f'{signal_type.title()} Wave: Original vs Fourier Series Reconstruction')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Fourier coefficients magnitude spectrum
        harmonics = np.arange(1, n_harmonics + 1)
        magnitude_spectrum = np.sqrt(np.array(coeffs['an'])**2 + np.array(coeffs['bn'])**2)
        
        # Include DC component if present
        if coeffs['a0'] != 0:
            freq_axis = np.concatenate(([0], harmonics * frequency))
            mag_axis = np.concatenate(([coeffs['a0']], magnitude_spectrum))
        else:
            freq_axis = harmonics * frequency
            mag_axis = magnitude_spectrum
        
        axes[0, 1].stem(freq_axis, mag_axis, basefmt='b-')
        axes[0, 1].set_xlabel('Frequency (Hz)')
        axes[0, 1].set_ylabel('Magnitude')
        axes[0, 1].set_title('Fourier Series Coefficients (Magnitude Spectrum)')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Phase spectrum
        phase_spectrum = np.arctan2(coeffs['bn'], coeffs['an'])
        axes[1, 0].stem(harmonics * frequency, phase_spectrum, basefmt='g-')
        axes[1, 0].set_xlabel('Frequency (Hz)')
        axes[1, 0].set_ylabel('Phase (radians)')
        axes[1, 0].set_title('Fourier Series Coefficients (Phase Spectrum)')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Individual harmonic contributions
        omega = 2 * np.pi * frequency
        colors = plt.cm.viridis(np.linspace(0, 1, min(5, n_harmonics)))
        
        for i in range(min(5, n_harmonics)):  # Show first 5 harmonics
            n = i + 1
            an = coeffs['an'][i]
            bn = coeffs['bn'][i]
            harmonic = an * np.cos(n * omega * t_plot) + bn * np.sin(n * omega * t_plot)
            axes[1, 1].plot(t_plot, harmonic, color=colors[i], linewidth=1.5, 
                           label=f'Harmonic {n} ({n*frequency:.1f} Hz)')
        
        axes[1, 1].set_xlabel('Time (s)')
        axes[1, 1].set_ylabel('Amplitude')
        axes[1, 1].set_title('Individual Harmonic Components')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Calculate and display error
        error = np.mean((original - reconstructed)**2)
        print(f"Mean Square Error between original and reconstructed signal: {error:.6f}")
        print(f"Number of harmonics used: {n_harmonics}")
        
        return original, reconstructed

# Create Fourier Series Visualizer
fs_viz = FourierSeriesVisualizer()
print("Fourier Series Visualizer initialized")

In [ ]:
# Interactive Fourier Series Visualization
def interactive_fourier_series(signal_type='square', frequency=1.0, n_harmonics=10):
    """Interactive Fourier series visualization"""
    return fs_viz.plot_fourier_series_analysis(signal_type, frequency, n_harmonics)

# Create interactive widget for Fourier Series
fourier_series_widget = interactive(
    interactive_fourier_series,
    signal_type=widgets.Dropdown(
        options=['square', 'sawtooth', 'triangle', 'half_rectified', 'full_rectified'],
        value='square',
        description='Signal Type:'
    ),
    frequency=widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Frequency (Hz):'),
    n_harmonics=widgets.IntSlider(value=10, min=1, max=50, step=1, description='Harmonics:')
)

display.display(fourier_series_widget)

In [ ]:
# ===================================================================
# TOPIC 2: FOURIER TRANSFORM SPECTRUM ANALYSIS
# ===================================================================

class FourierTransformAnalyzer:
    def __init__(self, sample_rate=44100):
        self.sample_rate = sample_rate
    
    def generate_test_signal(self, duration=2.0, frequencies=[440, 880, 1320], 
                           amplitudes=None, phases=None, noise_level=0.0):
        """Generate test signals for Fourier transform analysis"""
        if amplitudes is None:
            amplitudes = [1.0] * len(frequencies)
        if phases is None:
            phases = [0.0] * len(frequencies)
        
        t = np.linspace(0, duration, int(self.sample_rate * duration), False)
        signal = np.zeros_like(t)
        
        for freq, amp, phase in zip(frequencies, amplitudes, phases):
            signal += amp * np.sin(2 * np.pi * freq * t + phase)
        
        if noise_level > 0:
            noise = np.random.normal(0, noise_level, len(signal))
            signal += noise
        
        return signal, t
    
    def fourier_transform_analysis(self, signal, window_type='hann', freq_resolution=None):
        """Perform comprehensive Fourier transform analysis"""
        
        # Apply window function
        if window_type == 'hann':
            window = np.hanning(len(signal))
        elif window_type == 'hamming':
            window = np.hamming(len(signal))
        elif window_type == 'blackman':
            window = np.blackman(len(signal))
        elif window_type == 'kaiser':
            window = np.kaiser(len(signal), beta=8.6)
        else:  # rectangular
            window = np.ones(len(signal))
        
        windowed_signal = signal * window
        
        # Compute FFT
        fft_result = fft(windowed_signal)
        frequencies = fftfreq(len(signal), 1/self.sample_rate)
        
        # Magnitude and phase spectra
        magnitude = np.abs(fft_result)
        phase = np.angle(fft_result)
        
        # Power spectral density
        psd = magnitude**2 / (self.sample_rate * len(signal))
        
        # Return only positive frequencies
        positive_freq_mask = frequencies >= 0
        
        return {
            'frequencies': frequencies[positive_freq_mask],
            'magnitude': magnitude[positive_freq_mask],
            'phase': phase[positive_freq_mask],
            'psd': psd[positive_freq_mask],
            'windowed_signal': windowed_signal,
            'window': window
        }
    
    def plot_spectrum_analysis(self, signal, t, window_type='hann', freq_limit=5000):
        """Plot comprehensive spectrum analysis"""
        
        # Perform Fourier transform analysis
        analysis = self.fourier_transform_analysis(signal, window_type)
        
        # Create subplots
        fig, axes = plt.subplots(3, 2, figsize=(16, 14))
        
        # 1. Time domain signal and windowed signal
        axes[0, 0].plot(t, signal, 'b-', linewidth=1, label='Original Signal')
        axes[0, 0].plot(t, analysis['windowed_signal'], 'r-', linewidth=1, label=f'Windowed Signal ({window_type})')
        axes[0, 0].set_xlabel('Time (s)')
        axes[0, 0].set_ylabel('Amplitude')
        axes[0, 0].set_title('Time Domain Signal')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Window function
        axes[0, 1].plot(analysis['window'], 'g-', linewidth=2)
        axes[0, 1].set_xlabel('Sample Number')
        axes[0, 1].set_ylabel('Amplitude')
        axes[0, 1].set_title(f'Window Function: {window_type.title()}')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Apply frequency limit for visualization
        freq_mask = analysis['frequencies'] <= freq_limit
        freqs_plot = analysis['frequencies'][freq_mask]
        mag_plot = analysis['magnitude'][freq_mask]
        phase_plot = analysis['phase'][freq_mask]
        psd_plot = analysis['psd'][freq_mask]
        
        # 3. Magnitude spectrum (linear scale)
        axes[1, 0].plot(freqs_plot, mag_plot, 'b-', linewidth=1.5)
        axes[1, 0].set_xlabel('Frequency (Hz)')
        axes[1, 0].set_ylabel('Magnitude')
        axes[1, 0].set_title('Magnitude Spectrum (Linear Scale)')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Magnitude spectrum (dB scale)
        mag_db = 20 * np.log10(mag_plot + 1e-10)
        axes[1, 1].plot(freqs_plot, mag_db, 'r-', linewidth=1.5)
        axes[1, 1].set_xlabel('Frequency (Hz)')
        axes[1, 1].set_ylabel('Magnitude (dB)')
        axes[1, 1].set_title('Magnitude Spectrum (dB Scale)')
        axes[1, 1].grid(True, alpha=0.3)
        
        # 5. Phase spectrum
        axes[2, 0].plot(freqs_plot, phase_plot, 'g-', linewidth=1.5)
        axes[2, 0].set_xlabel('Frequency (Hz)')
        axes[2, 0].set_ylabel('Phase (radians)')
        axes[2, 0].set_title('Phase Spectrum')
        axes[2, 0].grid(True, alpha=0.3)
        
        # 6. Power Spectral Density
        axes[2, 1].semilogy(freqs_plot, psd_plot, 'm-', linewidth=1.5)
        axes[2, 1].set_xlabel('Frequency (Hz)')
        axes[2, 1].set_ylabel('PSD (V²/Hz)')
        axes[2, 1].set_title('Power Spectral Density')
        axes[2, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return analysis

# Create Fourier Transform Analyzer
ft_analyzer = FourierTransformAnalyzer()
print("Fourier Transform Analyzer initialized")

In [ ]:
# Interactive Fourier Transform Analysis
def interactive_spectrum_analysis(freq1=440, freq2=880, freq3=1320, 
                                amp1=1.0, amp2=0.5, amp3=0.25,
                                noise_level=0.0, window_type='hann'):
    """Interactive spectrum analysis"""
    
    # Generate test signal
    signal, t = ft_analyzer.generate_test_signal(
        duration=2.0,
        frequencies=[freq1, freq2, freq3],
        amplitudes=[amp1, amp2, amp3],
        noise_level=noise_level
    )
    
    # Perform analysis
    analysis = ft_analyzer.plot_spectrum_analysis(signal, t, window_type)
    
    # Save and play audio
    audio_path = "./spectrum_test.wav"
    write_wav(audio_path, signal, ft_analyzer.sample_rate)
    
    print("\nGenerated Test Signal:")
    display.display(display.Audio(filename=audio_path))
    
    return signal, analysis

# Create interactive widget for Spectrum Analysis
spectrum_widget = interactive(
    interactive_spectrum_analysis,
    freq1=widgets.IntSlider(value=440, min=100, max=2000, step=10, description='Freq 1 (Hz):'),
    freq2=widgets.IntSlider(value=880, min=100, max=2000, step=10, description='Freq 2 (Hz):'),
    freq3=widgets.IntSlider(value=1320, min=100, max=2000, step=10, description='Freq 3 (Hz):'),
    amp1=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='Amp 1:'),
    amp2=widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Amp 2:'),
    amp3=widgets.FloatSlider(value=0.25, min=0.0, max=2.0, step=0.1, description='Amp 3:'),
    noise_level=widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description='Noise Level:'),
    window_type=widgets.Dropdown(
        options=['rectangular', 'hann', 'hamming', 'blackman', 'kaiser'],
        value='hann',
        description='Window:'
    )
)

display.display(spectrum_widget)

In [ ]:
# ===================================================================
# TOPIC 3: DISCRETE FOURIER TRANSFORM (DFT) IMPLEMENTATION
# ===================================================================

class DFTImplementation:
    def __init__(self):
        pass
    
    def dft_naive(self, x):
        """Naive DFT implementation - O(N²) complexity"""
        N = len(x)
        X = np.zeros(N, dtype=complex)
        
        for k in range(N):
            for n in range(N):
                X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
        
        return X
    
    def dft_matrix(self, x):
        """DFT using matrix multiplication"""
        N = len(x)
        n = np.arange(N)
        k = n.reshape((N, 1))
        M = np.exp(-2j * np.pi * k * n / N)
        return np.dot(M, x)
    
    def dft_windowed(self, signal, window_size, overlap=0.5):
        """Windowed DFT for time-frequency analysis"""
        N = len(signal)
        hop_size = int(window_size * (1 - overlap))
        
        # Calculate number of windows
        num_windows = (N - window_size) // hop_size + 1
        
        # Prepare output arrays
        time_centers = []
        frequency_content = []
        
        # Apply Hann window
        window = np.hanning(window_size)
        
        # The DFT matrix depends only on the window size, so build it once and
        # reuse it for every frame (same O(N^2) maths as dft_naive, without the Python loops)
        n = np.arange(window_size)
        M = np.exp(-2j * np.pi * np.outer(n, n) / window_size)
        
        for i in range(num_windows):
            start = i * hop_size
            end = start + window_size
            
            if end <= N:
                # Extract windowed segment
                segment = signal[start:end] * window
                
                # Compute DFT
                X = M @ segment
                
                # Store results
                time_centers.append((start + end) / 2)
                frequency_content.append(X)
        
        return np.array(time_centers), np.array(frequency_content)
    
    def plot_dft_analysis(self, signal, sample_rate, window_size=1024, overlap=0.5):
        """Comprehensive DFT analysis visualization"""
        
        # Compute DFT of entire signal. An O(N^2) DFT of a 2 s signal (88,200 samples)
        # is ~7.8e9 operations, so use the FFT here. Topic 4 compares the two on short signals.
        X_full = np.fft.fft(signal)
        freqs_full = np.fft.fftfreq(len(signal), 1/sample_rate)
        
        # Compute windowed DFT
        time_centers, X_windowed = self.dft_windowed(signal, window_size, overlap)
        freqs_windowed = np.fft.fftfreq(window_size, 1/sample_rate)
        
        # Create subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Time domain signal
        t = np.linspace(0, len(signal)/sample_rate, len(signal))
        axes[0, 0].plot(t, signal, 'b-', linewidth=1)
        axes[0, 0].set_xlabel('Time (s)')
        axes[0, 0].set_ylabel('Amplitude')
        axes[0, 0].set_title('Original Signal')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Full signal DFT magnitude
        positive_mask = freqs_full >= 0
        freq_limit = min(5000, sample_rate // 2)
        freq_mask = (freqs_full >= 0) & (freqs_full <= freq_limit)
        
        axes[0, 1].plot(freqs_full[freq_mask], np.abs(X_full[freq_mask]), 'r-', linewidth=1.5)
        axes[0, 1].set_xlabel('Frequency (Hz)')
        axes[0, 1].set_ylabel('Magnitude')
        axes[0, 1].set_title('DFT Magnitude Spectrum (Full Signal)')
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Full signal DFT phase
        axes[0, 2].plot(freqs_full[freq_mask], np.angle(X_full[freq_mask]), 'g-', linewidth=1.5)
        axes[0, 2].set_xlabel('Frequency (Hz)')
        axes[0, 2].set_ylabel('Phase (radians)')
        axes[0, 2].set_title('DFT Phase Spectrum (Full Signal)')
        axes[0, 2].grid(True, alpha=0.3)
        
        # 4. Windowed DFT spectrogram
        if len(X_windowed) > 0:
            # Create spectrogram data
            freq_mask_win = (freqs_windowed >= 0) & (freqs_windowed <= freq_limit)
            freqs_plot = freqs_windowed[freq_mask_win]
            
            spectrogram_data = np.abs(X_windowed[:, freq_mask_win]).T
            
            # Convert time centers to seconds
            time_centers_sec = np.array(time_centers) / sample_rate
            
            im = axes[1, 0].pcolormesh(time_centers_sec, freqs_plot, 
                                     20 * np.log10(spectrogram_data + 1e-10),
                                     shading='gouraud', cmap='viridis')
            axes[1, 0].set_xlabel('Time (s)')
            axes[1, 0].set_ylabel('Frequency (Hz)')
            axes[1, 0].set_title(f'DFT Spectrogram (Window: {window_size}, Overlap: {overlap*100:.0f}%)')
            plt.colorbar(im, ax=axes[1, 0], label='Magnitude (dB)')
        
        # 5. Window visualization
        window = np.hanning(window_size)
        axes[1, 1].plot(window, 'purple', linewidth=2)
        axes[1, 1].set_xlabel('Sample Number')
        axes[1, 1].set_ylabel('Amplitude')
        axes[1, 1].set_title(f'Analysis Window (Size: {window_size})')
        axes[1, 1].grid(True, alpha=0.3)
        
        # 6. DFT parameters visualization
        axes[1, 2].text(0.1, 0.8, f'Signal Length: {len(signal)} samples', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.7, f'Sample Rate: {sample_rate} Hz', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.6, f'Duration: {len(signal)/sample_rate:.2f} s', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.5, f'Window Size: {window_size} samples', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.4, f'Overlap: {overlap*100:.0f}%', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.3, f'Frequency Resolution: {sample_rate/window_size:.2f} Hz', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].text(0.1, 0.2, f'Number of Windows: {len(X_windowed)}', transform=axes[1, 2].transAxes, fontsize=12)
        axes[1, 2].set_xlim(0, 1)
        axes[1, 2].set_ylim(0, 1)
        axes[1, 2].set_title('DFT Analysis Parameters')
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return X_full, X_windowed

# Create DFT Implementation
dft_impl = DFTImplementation()
print("DFT Implementation ready")

In [ ]:
# Interactive DFT Analysis
def interactive_dft_analysis(frequency=440, amplitude=1.0, duration=2.0, 
                           window_size=1024, overlap=0.5, noise_level=0.0):
    """Interactive DFT analysis with parameter control"""
    
    # Generate test signal
    sample_rate = 44100
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Create a more complex signal for interesting analysis
    signal = (amplitude * np.sin(2 * np.pi * frequency * t) +
              0.5 * amplitude * np.sin(2 * np.pi * 2 * frequency * t) +
              0.25 * amplitude * np.sin(2 * np.pi * 3 * frequency * t))
    
    # Add noise if specified
    if noise_level > 0:
        noise = np.random.normal(0, noise_level, len(signal))
        signal += noise
    
    # Perform DFT analysis
    X_full, X_windowed = dft_impl.plot_dft_analysis(signal, sample_rate, window_size, overlap)
    
    # Save and play audio
    audio_path = "./dft_test.wav"
    write_wav(audio_path, signal, sample_rate)
    
    print("\nGenerated DFT Test Signal:")
    display.display(display.Audio(filename=audio_path))
    
    return signal, X_full, X_windowed

# Create interactive widget for DFT Analysis
dft_widget = interactive(
    interactive_dft_analysis,
    frequency=widgets.IntSlider(value=440, min=100, max=1000, step=10, description='Base Freq (Hz):'),
    amplitude=widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Amplitude:'),
    duration=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='Duration (s):'),
    window_size=widgets.Dropdown(
        options=[256, 512, 1024, 2048, 4096],
        value=1024,
        description='Window Size:'
    ),
    overlap=widgets.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.1, description='Overlap:'),
    noise_level=widgets.FloatSlider(value=0.0, min=0.0, max=0.3, step=0.01, description='Noise Level:')
)

display.display(dft_widget)

In [ ]:
# ===================================================================
# TOPIC 4: FAST FOURIER TRANSFORM (FFT) COMPARISON
# ===================================================================

class FFTComparison:
    def __init__(self):
        self.dft_impl = DFTImplementation()
    
    def time_comparison(self, signal_lengths=[64, 128, 256, 512, 1024, 2048]):
        """Compare computation time between DFT and FFT"""
        
        dft_times = []
        fft_times = []
        
        print("Timing Comparison: DFT vs FFT")
        print("=" * 40)
        
        for N in signal_lengths:
            # Generate test signal
            x = np.random.randn(N)
            
            # Time DFT
            start_time = time.time()
            X_dft = self.dft_impl.dft_naive(x)
            dft_time = time.time() - start_time
            dft_times.append(dft_time)
            
            # Time FFT
            start_time = time.time()
            X_fft = np.fft.fft(x)
            fft_time = time.time() - start_time
            fft_times.append(fft_time)
            
            # Check accuracy
            error = np.max(np.abs(X_dft - X_fft))
            
            print(f"N={N:4d}: DFT={dft_time*1000:6.2f}ms, FFT={fft_time*1000:6.2f}ms, "
                  f"Speedup={dft_time/fft_time:6.1f}x, Error={error:.2e}")
        
        return signal_lengths, dft_times, fft_times
    
    def plot_timing_comparison(self, signal_lengths, dft_times, fft_times):
        """Plot timing comparison between DFT and FFT"""
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Linear scale comparison
        axes[0].plot(signal_lengths, np.array(dft_times) * 1000, 'ro-', linewidth=2, markersize=8, label='DFT (Naive)')
        axes[0].plot(signal_lengths, np.array(fft_times) * 1000, 'bo-', linewidth=2, markersize=8, label='FFT')
        axes[0].set_xlabel('Signal Length (N)')
        axes[0].set_ylabel('Computation Time (ms)')
        axes[0].set_title('DFT vs FFT: Computation Time (Linear Scale)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Log scale comparison
        axes[1].loglog(signal_lengths, np.array(dft_times) * 1000, 'ro-', linewidth=2, markersize=8, label='DFT O(N²)')
        axes[1].loglog(signal_lengths, np.array(fft_times) * 1000, 'bo-', linewidth=2, markersize=8, label='FFT O(N log N)')
        
        # Add theoretical complexity lines
        N_theory = np.array(signal_lengths)
        dft_theory = (N_theory**2) / (64**2) * dft_times[0] * 1000  # Normalize to first point
        fft_theory = (N_theory * np.log2(N_theory)) / (64 * np.log2(64)) * fft_times[0] * 1000
        
        axes[1].loglog(signal_lengths, dft_theory, 'r--', alpha=0.7, label='Theoretical O(N²)')
        axes[1].loglog(signal_lengths, fft_theory, 'b--', alpha=0.7, label='Theoretical O(N log N)')
        
        axes[1].set_xlabel('Signal Length (N)')
        axes[1].set_ylabel('Computation Time (ms)')
        axes[1].set_title('DFT vs FFT: Computation Time (Log Scale)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Calculate and display speedup factors
        speedups = np.array(dft_times) / np.array(fft_times)
        print("\nSpeedup Factors (DFT time / FFT time):")
        for N, speedup in zip(signal_lengths, speedups):
            print(f"N={N:4d}: {speedup:6.1f}x faster")
    
    def accuracy_comparison(self, signal_length=1024):
        """Compare accuracy between different FFT implementations"""
        
        # Generate test signal with known spectral content
        sample_rate = 44100
        duration = signal_length / sample_rate
        t = np.linspace(0, duration, signal_length, False)
        
        # Multi-tone signal
        frequencies = [440, 880, 1320]
        amplitudes = [1.0, 0.5, 0.25]
        phases = [0, np.pi/4, np.pi/2]
        
        signal = np.zeros_like(t)
        for freq, amp, phase in zip(frequencies, amplitudes, phases):
            signal += amp * np.sin(2 * np.pi * freq * t + phase)
        
        # Compute using different methods
        X_dft_naive = self.dft_impl.dft_naive(signal)
        X_dft_matrix = self.dft_impl.dft_matrix(signal)
        X_fft_numpy = np.fft.fft(signal)
        X_fft_scipy = fft(signal)
        
        # Compare results
        freq_axis = np.fft.fftfreq(signal_length, 1/sample_rate)
        positive_mask = freq_axis >= 0
        freq_plot = freq_axis[positive_mask]
        
        # Create comparison plot
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # Magnitude comparison
        axes[0, 0].plot(freq_plot, np.abs(X_dft_naive[positive_mask]), 'r-', linewidth=2, label='DFT Naive')
        axes[0, 0].plot(freq_plot, np.abs(X_fft_numpy[positive_mask]), 'b--', linewidth=2, label='NumPy FFT')
        axes[0, 0].set_xlabel('Frequency (Hz)')
        axes[0, 0].set_ylabel('Magnitude')
        axes[0, 0].set_title('Magnitude Spectrum Comparison')
        axes[0, 0].set_xlim(0, 2000)
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Phase comparison
        axes[0, 1].plot(freq_plot, np.angle(X_dft_naive[positive_mask]), 'r-', linewidth=2, label='DFT Naive')
        axes[0, 1].plot(freq_plot, np.angle(X_fft_numpy[positive_mask]), 'b--', linewidth=2, label='NumPy FFT')
        axes[0, 1].set_xlabel('Frequency (Hz)')
        axes[0, 1].set_ylabel('Phase (radians)')
        axes[0, 1].set_title('Phase Spectrum Comparison')
        axes[0, 1].set_xlim(0, 2000)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Error analysis
        error_matrix_vs_naive = np.abs(X_dft_matrix - X_dft_naive)
        error_numpy_vs_naive = np.abs(X_fft_numpy - X_dft_naive)
        error_scipy_vs_naive = np.abs(X_fft_scipy - X_dft_naive)
        
        axes[1, 0].semilogy(freq_plot, error_matrix_vs_naive[positive_mask], 'g-', linewidth=2, label='Matrix vs Naive')
        axes[1, 0].semilogy(freq_plot, error_numpy_vs_naive[positive_mask], 'b-', linewidth=2, label='NumPy FFT vs Naive')
        axes[1, 0].semilogy(freq_plot, error_scipy_vs_naive[positive_mask], 'm-', linewidth=2, label='SciPy FFT vs Naive')
        axes[1, 0].set_xlabel('Frequency (Hz)')
        axes[1, 0].set_ylabel('Absolute Error')
        axes[1, 0].set_title('Accuracy Comparison (vs Naive DFT)')
        axes[1, 0].set_xlim(0, 2000)
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Summary statistics
        max_errors = {
            'Matrix vs Naive': np.max(error_matrix_vs_naive),
            'NumPy FFT vs Naive': np.max(error_numpy_vs_naive),
            'SciPy FFT vs Naive': np.max(error_scipy_vs_naive)
        }
        
        y_pos = 0.8
        axes[1, 1].text(0.1, y_pos, 'Maximum Absolute Errors:', transform=axes[1, 1].transAxes, fontsize=14, weight='bold')
        y_pos -= 0.1
        
        for method, error in max_errors.items():
            axes[1, 1].text(0.1, y_pos, f'{method}: {error:.2e}', transform=axes[1, 1].transAxes, fontsize=12)
            y_pos -= 0.08
        
        axes[1, 1].text(0.1, y_pos-0.05, f'Signal Length: {signal_length}', transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].text(0.1, y_pos-0.13, f'Test Frequencies: {frequencies} Hz', transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].text(0.1, y_pos-0.21, f'Test Amplitudes: {amplitudes}', transform=axes[1, 1].transAxes, fontsize=12)
        
        axes[1, 1].set_xlim(0, 1)
        axes[1, 1].set_ylim(0, 1)
        axes[1, 1].set_title('Accuracy Summary')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print("Accuracy Analysis:")
        print("=" * 40)
        for method, error in max_errors.items():
            print(f"{method}: Max error = {error:.2e}")
        
        return max_errors

# Create FFT Comparison instance
fft_comparison = FFTComparison()
print("FFT Comparison tools ready")

In [ ]:
# Run comprehensive FFT vs DFT comparison
print("Running FFT vs DFT Timing Comparison...")
signal_lengths, dft_times, fft_times = fft_comparison.time_comparison()
fft_comparison.plot_timing_comparison(signal_lengths, dft_times, fft_times)

print("\n" + "="*60)
print("Running Accuracy Comparison...")
accuracy_results = fft_comparison.accuracy_comparison(1024)

In [ ]:
# Educational Examples and Demonstrations

print("="*80)
print("FREQUENCY DOMAIN ANALYSIS - EDUCATIONAL EXAMPLES")
print("="*80)

# Example 1: Window Functions Effect on Spectrum
print("\n1. EFFECT OF WINDOW FUNCTIONS ON SPECTRUM ANALYSIS")
print("-" * 50)

def demonstrate_window_effects():
    """Demonstrate how different window functions affect spectrum analysis"""
    
    # Generate a signal with closely spaced frequencies
    sample_rate = 44100
    duration = 1.0
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Two closely spaced sine waves
    freq1, freq2 = 1000, 1050  # 50 Hz apart
    signal = np.sin(2 * np.pi * freq1 * t) + np.sin(2 * np.pi * freq2 * t)
    
    window_types = ['rectangular', 'hann', 'hamming', 'blackman']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for i, window_type in enumerate(window_types):
        analysis = ft_analyzer.fourier_transform_analysis(signal, window_type)
        
        freq_mask = (analysis['frequencies'] >= 800) & (analysis['frequencies'] <= 1200)
        freqs_plot = analysis['frequencies'][freq_mask]
        mag_plot = analysis['magnitude'][freq_mask]
        
        axes[i].plot(freqs_plot, 20 * np.log10(mag_plot + 1e-10), linewidth=2)
        axes[i].set_xlabel('Frequency (Hz)')
        axes[i].set_ylabel('Magnitude (dB)')
        axes[i].set_title(f'{window_type.title()} Window')
        axes[i].grid(True, alpha=0.3)
        axes[i].axvline(freq1, color='r', linestyle='--', alpha=0.7, label=f'{freq1} Hz')
        axes[i].axvline(freq2, color='r', linestyle='--', alpha=0.7, label=f'{freq2} Hz')
        axes[i].legend()
    
    plt.suptitle('Effect of Window Functions on Frequency Resolution\n(Two Sine Waves: 1000 Hz and 1050 Hz)', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Save and play the test signal
    audio_path = "./window_test.wav"
    write_wav(audio_path, signal, sample_rate)
    
    print("Test signal (1000 Hz + 1050 Hz):")
    display.display(display.Audio(filename=audio_path))

demonstrate_window_effects()

# Example 2: Frequency Resolution vs Time Resolution Trade-off
print("\n2. TIME-FREQUENCY RESOLUTION TRADE-OFF")
print("-" * 50)

def demonstrate_resolution_tradeoff():
    """Demonstrate the trade-off between time and frequency resolution"""
    
    # Generate a signal with time-varying frequency content
    sample_rate = 44100
    duration = 3.0
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Create a signal with different frequency content in different time segments
    test_signal = np.zeros_like(t)  # Changed variable name from 'signal' to 'test_signal'
    
    # Segment 1: 440 Hz (0-1 sec)
    mask1 = (t >= 0) & (t < 1)
    test_signal[mask1] = np.sin(2 * np.pi * 440 * t[mask1])
    
    # Segment 2: 880 Hz (1-2 sec)
    mask2 = (t >= 1) & (t < 2)
    test_signal[mask2] = np.sin(2 * np.pi * 880 * t[mask2])
    
    # Segment 3: Chirp (2-3 sec)
    mask3 = (t >= 2) & (t < 3)
    t_chirp = t[mask3] - 2  # Reset time for chirp
    test_signal[mask3] = np.sin(2 * np.pi * (1000 + 500 * t_chirp) * t_chirp)
    
    # Analyze with different window sizes
    window_sizes = [256, 1024, 4096]
    
    fig, axes = plt.subplots(len(window_sizes), 2, figsize=(16, 12))
    
    for i, window_size in enumerate(window_sizes):
        # Compute spectrogram - now using scipy.signal.spectrogram correctly
        f, t_spec, Sxx = signal.spectrogram(test_signal, sample_rate, nperseg=window_size, noverlap=window_size//2)
        
        # Plot spectrogram
        im = axes[i, 0].pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud')
        axes[i, 0].set_ylabel('Frequency (Hz)')
        axes[i, 0].set_title(f'Window Size: {window_size} (Freq Res: {sample_rate/window_size:.1f} Hz)')
        axes[i, 0].set_ylim(0, 2000)
        
        # Plot time resolution visualization
        time_res = window_size / sample_rate
        freq_res = sample_rate / window_size
        
        axes[i, 1].text(0.1, 0.8, f'Window Size: {window_size} samples', transform=axes[i, 1].transAxes, fontsize=12)
        axes[i, 1].text(0.1, 0.7, f'Time Resolution: {time_res*1000:.1f} ms', transform=axes[i, 1].transAxes, fontsize=12)
        axes[i, 1].text(0.1, 0.6, f'Frequency Resolution: {freq_res:.1f} Hz', transform=axes[i, 1].transAxes, fontsize=12)
        axes[i, 1].text(0.1, 0.4, 'Trade-off:', transform=axes[i, 1].transAxes, fontsize=12, weight='bold')
        
        if window_size == 256:
            axes[i, 1].text(0.1, 0.3, '• Good time resolution', transform=axes[i, 1].transAxes, fontsize=11)
            axes[i, 1].text(0.1, 0.2, '• Poor frequency resolution', transform=axes[i, 1].transAxes, fontsize=11)
        elif window_size == 1024:
            axes[i, 1].text(0.1, 0.3, '• Balanced resolution', transform=axes[i, 1].transAxes, fontsize=11)
        else:
            axes[i, 1].text(0.1, 0.3, '• Poor time resolution', transform=axes[i, 1].transAxes, fontsize=11)
            axes[i, 1].text(0.1, 0.2, '• Good frequency resolution', transform=axes[i, 1].transAxes, fontsize=11)
        
        axes[i, 1].set_xlim(0, 1)
        axes[i, 1].set_ylim(0, 1)
        axes[i, 1].axis('off')
    
    axes[-1, 0].set_xlabel('Time (s)')
    
    plt.suptitle('Time-Frequency Resolution Trade-off in Spectrograms', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Save and play the test signal
    audio_path = "./resolution_test.wav"
    write_wav(audio_path, test_signal, sample_rate)
    
    print("Test signal (440 Hz → 880 Hz → Chirp):")
    display.display(display.Audio(filename=audio_path))

demonstrate_resolution_tradeoff()

print("\n" + "="*80)
print("FREQUENCY DOMAIN ANALYSIS COMPLETE!")
print("Key Concepts Demonstrated:")
print("• Fourier Series representation of periodic signals")
print("• Fourier Transform spectrum analysis with windowing")
print("• DFT implementation and windowed analysis")
print("• FFT efficiency compared to DFT")
print("• Window function effects on spectral analysis")
print("• Time-frequency resolution trade-offs")
print("="*80)

## Topic 5: Short-time Fourier transform (STFT)
A single DFT of a whole recording shows which frequencies are present, but not when. The STFT slides a window along the signal and takes a DFT of each frame:

$$X[m, k] = \sum_{n=0}^{N-1} x[n + mH]\; w[n]\; e^{-j 2\pi k n / N}$$

- $N$: frame length. It sets the frequency resolution $f_s / N$ (Example 2 above shows the time-frequency trade-off).
- $H$: hop size, the step between frames. Consecutive frames overlap by $N - H$ samples.
- $w[n]$: the analysis window (Topic 2 and Example 1 show how the window shapes leakage).

The spectrograms in this notebook are $|X[m, k]|^2$.

The STFT can be inverted: inverse-DFT each frame, then add the frames back at their original positions (**overlap-add**). Reconstruction is exact when the shifted windows add up to a constant, the **COLA** (constant overlap-add) condition. A Hann window meets it at 50% and 75% overlap.

Most spectral processing in later modules works this way: STFT, change the frames, inverse STFT. Noise reduction (`07_Audio_Restoration_and_Enhancement`), pitch shifting (`05_Audio_Effects`) and perceptual coding (`06_Psychoacoustics_and_Audio_Coding`) are all built on it.

In [ ]:
# ===================================================================
# TOPIC 5: SHORT-TIME FOURIER TRANSFORM (STFT)
# ===================================================================
fs = 44100
t = np.arange(2 * fs) / fs
x = signal.chirp(t, 200, 2.0, 4000) + 0.5 * np.sin(2 * np.pi * 1000 * t)

print("COLA condition (exact overlap-add reconstruction possible):")
for window, nperseg, noverlap in [('hann', 1024, 512), ('hann', 1024, 768), ('hann', 1024, 300), ('boxcar', 1024, 0)]:
    print(f"  {window:<7} N = {nperseg}, overlap {noverlap:4d} ({100 * noverlap / nperseg:3.0f}%): {signal.check_COLA(window, nperseg, noverlap)}")

f_stft, t_stft, Z = signal.stft(x, fs, window='hann', nperseg=1024, noverlap=768)
_, x_rec = signal.istft(Z, fs, window='hann', nperseg=1024, noverlap=768)
print(f"\nSTFT: {Z.shape[0]} frequency bins x {Z.shape[1]} frames")
print(f"Max reconstruction error after inverse STFT: {np.max(np.abs(x_rec[:len(x)] - x)):.1e}")

### Processing in the STFT domain: a spectral gate
A first taste of STFT processing. Two tones are buried in white noise. In the STFT, the tones put their energy into a few bins, while the noise spreads thinly over all of them. Keeping only the bins well above the typical (median) bin level, and zeroing the rest, removes most of the noise.

This hard gate is the crudest form of spectral noise reduction. Where it lets isolated noise bins through, they sound like brief random tones ("musical noise"). `07_Audio_Restoration_and_Enhancement` covers better methods.

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(2 * fs) / fs
tones = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.2 * np.sin(2 * np.pi * 660 * t)
noisy = tones + rng.normal(0, 0.1, len(t))

f_stft, t_stft, Z = signal.stft(noisy, fs, nperseg=2048, noverlap=1536)
threshold = 4 * np.median(np.abs(Z))
_, cleaned = signal.istft(Z * (np.abs(Z) > threshold), fs, nperseg=2048, noverlap=1536)
cleaned = cleaned[:len(noisy)]

def snr_db(clean, test):
    return 10 * np.log10(np.mean(clean**2) / np.mean((test - clean)**2))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, y, name in [(axes[0], noisy, 'Noisy'), (axes[1], cleaned, 'After spectral gate')]:
    f_s, t_s, Sxx = signal.spectrogram(y, fs, nperseg=2048)
    ax.pcolormesh(t_s, f_s, 10 * np.log10(Sxx + 1e-15), shading='auto', vmin=-120, vmax=-20)
    ax.set_ylim(0, 2000)
    ax.set_title(f'{name}: SNR {snr_db(tones, y):.1f} dB')
    ax.set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

print("Noisy:")
display.display(display.Audio(noisy, rate=fs))
print("After spectral gate:")
display.display(display.Audio(cleaned, rate=fs))